In [9]:
"""
═══════════════════════════════════════════════════════════════════
SLU INTERNATIONAL STUDENT APPLICATIONS - WEEK 2
Notebook 1: Data Cleaning & PostgreSQL Export (CORRECTED)
═══════════════════════════════════════════════════════════════════

Purpose: Load, clean, validate, and export data for PostgreSQL import

Author: Team 1
Date: February 2026
Week: 2 - Database Preparation

Key Improvements:
- Proper column name standardization from start
- ERD-aligned column selection
- Validation before export
- Clean, readable code structure

═══════════════════════════════════════════════════════════════════
"""

# ═══════════════════════════════════════════════════════════════════
# SECTION 1: SETUP & IMPORTS
# ═══════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from datetime import datetime
import warnings
import os

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ Libraries imported")
print(f"✓ Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# ═══════════════════════════════════════════════════════════════════
# SECTION 2: DEFINE ERD SCHEMA (SOURCE OF TRUTH)
# ═══════════════════════════════════════════════════════════════════

print("DEFINING POSTGRESQL SCHEMA")
print("═" * 70)

# These are your PostgreSQL table schemas - NEVER change these
ERD_COLUMNS = {
    'dim_applicant': [
        'reference_id', 'given_name', 'last_name', 'major', 'citizenship',
        'intake', 'application_source', 'sevis_id', 'email_id', 'slu_email',
        'application_date', 'is_global_grad', 'date_of_birth', 'phone_number',
        'city', 'region', 'postal', 'most_recent_released_decision', 'created_at'
    ],

    'dim_sevis': [
        'sevis_id', 'country_of_citizenship', 'gender', 'tuition_fees',
        'living_expenses', 'student_personal_funds', 'funds_from_school',
        'school_fund_type_standardized', 'funds_from_other_sources',
        'major_1_description', 'education_level', 'program_start_date',
        'program_end_date', 'sevis_status', 'class_of_admission'
    ],

   'dim_outreach': [
    'sevis_id', 'student_full_name', 'program_of_interest', 'term',
    'deposit_paid',  # ← Make sure this is here
    'accepted_admission', 'student_status',
    'citizenship_primary', 'application_agency_code', 'i20_in_slate',
    'i901_status', 'person_email', 'person_slu_email', 'appointment_date'
],

    'fact_connect_events': [
        'reference_id', 'deposit_status', 'i20_status', 'assigned',
        'event_timestamp', 'connect_created_at'
    ],

    'fact_student_status': [
        'reference_id', 'given_name', 'last_name', 'major', 'citizenship',
        'intake', 'application_source', 'sevis_id', 'deposit_status', 'i20_status',
        'student_status', 'has_deposit', 'has_i20', 'has_visa',
        'funds_from_school', 'school_fund_type_standardized', 'tuition_fees',
        'country_of_citizenship', 'program_start_date', 'program_end_date',
        'application_agency_code', 'application_date', 'latest_event_timestamp', 'assigned'
    ]
}

print(f"✓ ERD schema defined for {len(ERD_COLUMNS)} tables:")
for table, cols in ERD_COLUMNS.items():
    print(f"  • {table}: {len(cols)} columns")

print()

# ═══════════════════════════════════════════════════════════════════
# SECTION 3: HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════

def standardize_columns(df):
    """
    Convert column names to snake_case for PostgreSQL compatibility

    Examples:
    - 'Given Name' → 'given_name'
    - 'Student\'s_Personal_Funds' → 'students_personal_funds'
    - 'I-20_Status' → 'i_20_status'
    """
    df.columns = (df.columns
                  .str.replace("'", '', regex=False)
                  .str.replace('-', '_', regex=False)
                  .str.replace(' ', '_', regex=False)
                  .str.lower())
    return df


def convert_timestamps(df, timestamp_cols, unit='ms'):
    """
    Convert Unix timestamps (milliseconds) to datetime

    Parameters:
    -----------
    df : pd.DataFrame
    timestamp_cols : list
        Column names containing timestamps
    unit : str
        'ms' for milliseconds, 's' for seconds
    """
    for col in timestamp_cols:
        if col in df.columns:
            try:
                df[col] = pd.to_datetime(df[col], unit=unit, errors='coerce')
                print(f"  ✓ Converted {col} to datetime")
            except Exception as e:
                print(f"  ✗ Failed to convert {col}: {e}")
    return df


def convert_date_strings(df, date_cols, date_format='%m/%d/%Y'):
    """
    Convert string dates to datetime

    Parameters:
    -----------
    df : pd.DataFrame
    date_cols : list
        Column names containing date strings
    date_format : str
        Format string (default: MM/DD/YYYY)
    """
    for col in date_cols:
        if col in df.columns:
            try:
                df[col] = pd.to_datetime(df[col], format=date_format, errors='coerce')
                print(f"  ✓ Converted {col} to datetime")
            except Exception as e:
                print(f"  ✗ Failed to convert {col}: {e}")
    return df


def validate_for_postgres(df, table_name):
    """
    Validate dataframe is ready for PostgreSQL export

    Checks:
    1. Required columns present
    2. No duplicate primary keys
    3. Reasonable completeness

    Returns:
    --------
    bool : True if valid, False otherwise
    """

    if table_name not in ERD_COLUMNS:
        print(f"❌ Unknown table: {table_name}")
        return False

    required_cols = ERD_COLUMNS[table_name]
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        print(f"❌ {table_name} VALIDATION FAILED")
        print(f"   Missing columns: {missing_cols}")
        return False

    # Check primary key (first column is always PK)
    pk_col = required_cols[0]
    dupes = df[pk_col].duplicated().sum()

    if dupes > 0:
        print(f"❌ {table_name} VALIDATION FAILED")
        print(f"   Duplicate primary keys ({pk_col}): {dupes}")
        return False

    # Check completeness
    completeness = df[required_cols].count().sum() / (len(df) * len(required_cols)) * 100

    print(f"✓ {table_name} validation PASSED")
    print(f"  Rows: {len(df):,} | Columns: {len(required_cols)} | Completeness: {completeness:.1f}%")

    return True


def export_for_postgres(df, table_name, output_path='./'):
    """
    Export dataframe with ONLY ERD-defined columns

    Parameters:
    -----------
    df : pd.DataFrame
        Cleaned dataframe
    table_name : str
        Name matching ERD_COLUMNS key
    output_path : str
        Directory for CSV output

    Returns:
    --------
    pd.DataFrame : Exported dataframe (for verification)
    """

    if table_name not in ERD_COLUMNS:
        raise ValueError(f"❌ Table '{table_name}' not in ERD schema")

    # Get ERD columns
    required_cols = ERD_COLUMNS[table_name]

    # Check availability
    available = [col for col in required_cols if col in df.columns]
    missing = [col for col in required_cols if col not in df.columns]

    if missing:
        print(f"⚠ {table_name}: Missing columns {missing}")

    # Select and export
    df_export = df[available].copy()

    filename = f"{output_path}{table_name}.csv"
    df_export.to_csv(filename, index=False)

    file_size = os.path.getsize(filename) / 1024  # KB
    print(f"✓ Exported {table_name}.csv: {len(df_export):,} rows × {len(available)} cols ({file_size:.1f} KB)")

    return df_export


print("✓ Helper functions defined\n")

# ═══════════════════════════════════════════════════════════════════
# SECTION 4: LOAD RAW DATA
# ═══════════════════════════════════════════════════════════════════

from google.colab import files

print("LOADING RAW DATA")
print("═" * 70)
print("Please upload your 4 CSV files:")

uploaded = files.upload()

# Auto-detect files
def find_file(keyword, file_list):
    for f in file_list:
        if keyword.lower() in f.lower():
            return f
    return None

uploaded_files = list(uploaded.keys())

applicant_file = find_file('applicant', uploaded_files)
sevis_file = find_file('sevis', uploaded_files)
connect_file = find_file('connect', uploaded_files)
outreach_file = find_file('outreach', uploaded_files)

print(f"\n✓ File mapping:")
print(f"  Applicant → {applicant_file}")
print(f"  SEVIS     → {sevis_file}")
print(f"  Connect   → {connect_file}")
print(f"  Outreach  → {outreach_file}\n")

# Load CSVs
applicant_raw = pd.read_csv(applicant_file, low_memory=False)
sevis_raw = pd.read_csv(sevis_file, low_memory=False)
connect_raw = pd.read_csv(connect_file, low_memory=False)
outreach_raw = pd.read_csv(outreach_file, low_memory=False)

print("✓ Data loaded:")
print(f"  Applicant: {applicant_raw.shape[0]:,} rows × {applicant_raw.shape[1]} cols")
print(f"  SEVIS:     {sevis_raw.shape[0]:,} rows × {sevis_raw.shape[1]} cols")
print(f"  Connect:   {connect_raw.shape[0]:,} rows × {connect_raw.shape[1]} cols")
print(f"  Outreach:  {outreach_raw.shape[0]:,} rows × {outreach_raw.shape[1]} cols\n")

# ═══════════════════════════════════════════════════════════════════
# SECTION 5: STANDARDIZE COLUMN NAMES (CRITICAL!)
# ═══════════════════════════════════════════════════════════════════

print("STANDARDIZING COLUMN NAMES")
print("═" * 70)

applicant_raw = standardize_columns(applicant_raw)
sevis_raw = standardize_columns(sevis_raw)
connect_raw = standardize_columns(connect_raw)
outreach_raw = standardize_columns(outreach_raw)

print("✓ All column names converted to snake_case\n")

# ═══════════════════════════════════════════════════════════════════
# SECTION 5B: MAP COLUMN NAMES TO ERD SCHEMA
# ═══════════════════════════════════════════════════════════════════

print("MAPPING COLUMNS TO ERD SCHEMA")
print("═" * 70)

# Applicant - Rename to match ERD
applicant_raw.rename(columns={
    'official_university_email_address': 'slu_email'
}, inplace=True)
print("✓ Applicant columns mapped")

# Outreach - Rename to match ERD
outreach_raw.rename(columns={
    'citizenship': 'citizenship_primary',
    'i_20_in_slate': 'i20_in_slate',
    'i_901_status': 'i901_status',
    'person_slu_email': 'person_slu_email'  # Check if exists
}, inplace=True, errors='ignore')
print("✓ Outreach columns mapped")

print()

# ═══════════════════════════════════════════════════════════════════
# SECTION 6: DATE CONVERSIONS
# ═══════════════════════════════════════════════════════════════════

print("CONVERTING DATE COLUMNS")
print("═" * 70)

print("\n1. Applicant timestamps (milliseconds):")
applicant_raw = convert_timestamps(
    applicant_raw,
    ['received_at', 'created_at', 'modified_at'],
    unit='ms'
)
applicant_raw.rename(columns={'received_at': 'application_date'}, inplace=True)

print("\n2. Applicant date strings:")
applicant_raw = convert_date_strings(
    applicant_raw,
    ['date_of_birth', 'admit_date']
)

print("\n3. SEVIS date strings:")
sevis_raw = convert_date_strings(
    sevis_raw,
    ['status_change_date', 'date_of_birth', 'program_start_date',
     'program_end_date', 'visa_issuance_date', 'passport_expiration_date',
     'date_of_entry', 'i_901_transaction_date']
)

print("\n4. Connect timestamps:")
connect_raw = convert_timestamps(
    connect_raw,
    ['created_at', 'modified_at'],
    unit='ms'
)
connect_raw.rename(columns={'recieved_at': 'event_timestamp'}, inplace=True)
connect_raw['event_timestamp'] = pd.to_datetime(connect_raw['event_timestamp'],
                                                  format='%m/%d/%Y %H:%M', errors='coerce')
connect_raw.rename(columns={'created_at': 'connect_created_at'}, inplace=True)

print("\n✓ All date conversions complete\n")

# ═══════════════════════════════════════════════════════════════════
# SECTION 7: SEVIS CLEANING
# ═══════════════════════════════════════════════════════════════════

print("CLEANING SEVIS DATA")
print("═" * 70)

sevis_clean = sevis_raw.copy()

# Rename SEVIS columns to match ERD schema (cleaner names)


sevis_clean.rename(columns={
    'funds_from_this_school': 'funds_from_school',
    'students_personal_funds': 'student_personal_funds',
    'funds_from_other_sources': 'funds_from_other_sources'  # Keep as-is, just documenting
}, inplace=True)

print("✓ SEVIS column names aligned with ERD schema")

# Standardize fund types
print("\n1. Standardizing fund types:")

def standardize_fund_type(value):
    if pd.isna(value):
        return 'Unknown'
    value_str = str(value).strip().lower()

    if 'scholarship' in value_str and 'assistantship' in value_str:
        return 'Scholarship and Assistantship'
    elif 'graduate assistantship' in value_str:
        return 'Graduate Assistantship'
    elif 'teaching assistantship' in value_str:
        return 'Teaching Assistantship'
    elif 'research assistantship' in value_str:
        return 'Research Assistantship'
    elif 'assistantship' in value_str:
        return 'Assistantship'
    elif 'global graduate scholarship' in value_str:
        return 'Global Graduate Scholarship'
    elif 'scholarship' in value_str:
        return 'Scholarship'
    elif 'grant' in value_str:
        return 'Grant'
    elif 'fellowship' in value_str:
        return 'Fellowship'
    else:
        return 'Other'

if 'school_fund_type' in sevis_clean.columns:
    sevis_clean['school_fund_type_standardized'] = sevis_clean['school_fund_type'].apply(standardize_fund_type)
    print(f"  ✓ Standardized {sevis_clean['school_fund_type'].nunique()} variations to {sevis_clean['school_fund_type_standardized'].nunique()} categories")

print(f"\n✓ SEVIS cleaned: {len(sevis_clean):,} rows\n")

# ═══════════════════════════════════════════════════════════════════
# SECTION 8: APPLICANT DEDUPLICATION
# ═══════════════════════════════════════════════════════════════════

print("DEDUPLICATING APPLICANT DATA")
print("═" * 70)

print(f"Original: {len(applicant_raw):,} rows")

# Check for duplicates
dupes = applicant_raw['reference_id'].duplicated().sum()
print(f"Duplicates found: {dupes}")

if dupes > 0:
    # Remove duplicates, keep first occurrence
    applicant_clean = applicant_raw.drop_duplicates(subset=['reference_id'], keep='first')
    print(f"After deduplication: {len(applicant_clean):,} rows")
else:
    applicant_clean = applicant_raw.copy()
    print("✓ No duplicates found")

print()

# Load confirmed matches
confirmed = pd.read_csv('confirmed_matches_93.csv')
confirmed = confirmed[confirmed['Confirmed'] == 1].copy()
confirmed['visa_sevis_id'] = confirmed['visa_sevis_id'].astype(str).str.strip()

# Apply updates to applicant_clean
existing_sevis = set(applicant_clean['sevis_id'].dropna().astype(str).str.strip())
for _, row in confirmed.iterrows():
    sevis_id = row['visa_sevis_id']
    ref_id = row['applicant_reference_id']
    if sevis_id not in existing_sevis:
        mask = applicant_clean['reference_id'] == ref_id
        if mask.any() and pd.isna(applicant_clean.loc[mask, 'sevis_id'].iloc[0]):
            applicant_clean.loc[mask, 'sevis_id'] = sevis_id
            existing_sevis.add(sevis_id)

# ═══════════════════════════════════════════════════════════════════
# SECTION 9: CONNECT CLEANING & DEDUPLICATION
# ═══════════════════════════════════════════════════════════════════

print("CLEANING CONNECT DATA")
print("═" * 70)

connect_clean = connect_raw.copy()

# Rename columns to match ERD
connect_clean.rename(columns={
    'deposit_status': 'deposit_status',
    'i_20_status': 'i20_status'
}, inplace=True)

print(f"Total records: {len(connect_clean):,}")
print(f"Unique students: {connect_clean['reference_id'].nunique():,}")
print(f"Duplicate rate: {(1 - connect_clean['reference_id'].nunique() / len(connect_clean)) * 100:.1f}%")

# Create latest status view
connect_latest = (connect_clean
                  .sort_values('connect_created_at')
                  .groupby('reference_id', as_index=False)
                  .last())

print(f"\nLatest status per student: {len(connect_latest):,} rows")
print()

# ═══════════════════════════════════════════════════════════════════
# SECTION 10: OUTREACH CLEANING
# ═══════════════════════════════════════════════════════════════════

print("CLEANING OUTREACH DATA")
print("═" * 70)

outreach_clean = outreach_raw.copy()

# ═══════════════════════════════════════════════════════════════════
# SECTION 10: OUTREACH CLEANING - FINAL VERSION
# ═══════════════════════════════════════════════════════════════════

print("CLEANING OUTREACH DATA")
print("═" * 70)

outreach_clean = outreach_raw.copy()

# Rename columns to match ERD schema exactly
outreach_clean.rename(columns={
    'citizenship_(primary)': 'citizenship_primary',  # Remove parentheses
    'i_20_in_slate': 'i20_in_slate',                 # Remove underscore
    'i_901_status': 'i901_status',                   # Remove underscore
    'person_slu_email_address': 'person_slu_email'   # Shorten name
}, inplace=True)

print("✓ Column names aligned with ERD schema")

# Check for duplicates
dupes = outreach_clean['sevis_id'].duplicated().sum()
print(f"Duplicates found: {dupes}")

if dupes > 0:
    outreach_clean = outreach_clean.drop_duplicates(subset=['sevis_id'], keep='first')
    print(f"After deduplication: {len(outreach_clean):,} rows")
else:
    print("✓ No duplicates found")

# Count visa approved
if 'student_status' in outreach_clean.columns:
    visa_approved = outreach_clean['student_status'].str.contains('Visa approved', na=False, case=False).sum()
    print(f"Visa approved students: {visa_approved}")

print()

# ═══════════════════════════════════════════════════════════════════
# SECTION 11: BUILD MASTER TABLE (fact_student_status) - FINAL FIX
# ═══════════════════════════════════════════════════════════════════

print("BUILDING MASTER TABLE (fact_student_status)")
print("═" * 70)

# Start with applicant
master = applicant_clean.copy()

# Merge Connect (latest status)
print("\n1. Merging Connect data...")
connect_merge_cols = ['reference_id', 'deposit_status', 'i20_status', 'assigned', 'event_timestamp']
master = master.merge(
    connect_latest[connect_merge_cols],
    on='reference_id',
    how='left'
)
print(f"   ✓ After Connect merge: {len(master):,} rows")

# Merge SEVIS (INCLUDING funds_from_school!)
print("\n2. Merging SEVIS data...")
sevis_merge_cols = [
    'sevis_id',
    'country_of_citizenship',
    'funds_from_school',  # ← CRITICAL: Include this!
    'school_fund_type_standardized',
    'tuition_fees',
    'program_start_date',
    'program_end_date'
]

master = master.merge(
    sevis_clean[sevis_merge_cols],
    on='sevis_id',
    how='left'
)
print(f"   ✓ After SEVIS merge: {len(master):,} rows")

# Merge Outreach
print("\n3. Merging Outreach data...")
outreach_merge_cols = ['sevis_id', 'student_status']

# Add optional columns if they exist
for col in ['application_agency_code', 'citizenship_primary']:
    if col in outreach_clean.columns:
        outreach_merge_cols.append(col)

master = master.merge(
    outreach_clean[outreach_merge_cols],
    on='sevis_id',
    how='left',
    suffixes=('', '_outreach')
)
print(f"   ✓ After Outreach merge: {len(master):,} rows")

# Handle application_agency_code conflict
if 'application_agency_code_outreach' in master.columns:
    master['application_agency_code'] = master['application_agency_code_outreach'].fillna(
        master.get('application_agency_code', None)
    )
    master.drop(columns=['application_agency_code_outreach'], inplace=True, errors='ignore')

# Add derived flags
print("\n4. Adding derived flags...")
master['has_deposit'] = (master['deposit_status'] == 'Yes').astype(int)
master['has_i20'] = (master['i20_status'] == 'Yes').astype(int)
master['has_visa'] = (master['student_status'] == 'Visa approved').astype(int)

# Rename for clarity
master.rename(columns={
    'event_timestamp': 'latest_event_timestamp'
}, inplace=True)

fact_student_status = master.copy()

print(f"\n✓ Master table built: {len(fact_student_status):,} rows")
print(f"\nKey Metrics:")
print(f"  Students with deposit: {master['has_deposit'].sum():,} ({master['has_deposit'].sum()/len(master)*100:.1f}%)")
print(f"  Students with I-20:    {master['has_i20'].sum():,} ({master['has_i20'].sum()/len(master)*100:.1f}%)")
print(f"  Students with visa:    {master['has_visa'].sum():,} ({master['has_visa'].sum()/len(master)*100:.1f}%)")

print()

# ═══════════════════════════════════════════════════════════════════
# SECTION 12: VALIDATION
# ═══════════════════════════════════════════════════════════════════

print("VALIDATING DATA FOR POSTGRESQL")
print("═" * 70)
print()

validate_for_postgres(applicant_clean, 'dim_applicant')
validate_for_postgres(sevis_clean, 'dim_sevis')
validate_for_postgres(outreach_clean, 'dim_outreach')
validate_for_postgres(connect_latest, 'fact_connect_events')
validate_for_postgres(fact_student_status, 'fact_student_status')

print()

# ═══════════════════════════════════════════════════════════════════
# SECTION 13: EXPORT FOR POSTGRESQL
# ═══════════════════════════════════════════════════════════════════

print("EXPORTING CLEAN DATA FOR POSTGRESQL")
print("═" * 70)
print()

# Export all tables
dim_applicant_export = export_for_postgres(applicant_clean, 'dim_applicant')
dim_sevis_export = export_for_postgres(sevis_clean, 'dim_sevis')
dim_outreach_export = export_for_postgres(outreach_clean, 'dim_outreach')
fact_connect_export = export_for_postgres(connect_latest, 'fact_connect_events')
fact_status_export = export_for_postgres(fact_student_status, 'fact_student_status')

print()
print("═" * 70)
print("✓ ALL EXPORTS COMPLETE!")
print("═" * 70)
print("\nFiles ready for PostgreSQL import:")
print("  1. dim_applicant.csv")
print("  2. dim_sevis.csv")
print("  3. dim_outreach.csv")
print("  4. fact_connect_events.csv")
print("  5. fact_student_status.csv")
print("\nNext step: Import into pgAdmin using COPY command")

# Download files
from google.colab import files

print("\n" + "─" * 70)
print("DOWNLOADING FILES")
print("─" * 70)

for filename in ['dim_applicant.csv', 'dim_sevis.csv', 'dim_outreach.csv',
                 'fact_connect_events.csv', 'fact_student_status.csv']:
    files.download(filename)

print("\n✓ All files downloaded to your local machine!")

✓ Libraries imported
✓ Timestamp: 2026-02-13 16:07:55

DEFINING POSTGRESQL SCHEMA
══════════════════════════════════════════════════════════════════════
✓ ERD schema defined for 5 tables:
  • dim_applicant: 19 columns
  • dim_sevis: 15 columns
  • dim_outreach: 14 columns
  • fact_connect_events: 6 columns
  • fact_student_status: 24 columns

✓ Helper functions defined

LOADING RAW DATA
══════════════════════════════════════════════════════════════════════
Please upload your 4 CSV files:


Saving Mein Copy of Connect - Sheet1.csv to Mein Copy of Connect - Sheet1.csv
Saving Mein Copy of Outreach_Data - Sheet1.csv to Mein Copy of Outreach_Data - Sheet1.csv
Saving Mein Copy of SEVIS - Sheet1.csv to Mein Copy of SEVIS - Sheet1.csv
Saving Mien Copy of Applicant_Data - Sheet1.csv to Mien Copy of Applicant_Data - Sheet1.csv

✓ File mapping:
  Applicant → Mien Copy of Applicant_Data - Sheet1.csv
  SEVIS     → Mein Copy of SEVIS - Sheet1.csv
  Connect   → Mein Copy of Connect - Sheet1.csv
  Outreach  → Mein Copy of Outreach_Data - Sheet1.csv

✓ Data loaded:
  Applicant: 6,894 rows × 30 cols
  SEVIS:     3,852 rows × 123 cols
  Connect:   34,341 rows × 8 cols
  Outreach:  1,565 rows × 18 cols

STANDARDIZING COLUMN NAMES
══════════════════════════════════════════════════════════════════════
✓ All column names converted to snake_case

MAPPING COLUMNS TO ERD SCHEMA
══════════════════════════════════════════════════════════════════════
✓ Applicant columns mapped
✓ Outreach columns map

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ All files downloaded to your local machine!


In [3]:
# ═══════════════════════════════════════════════════════════════════
# EMERGENCY FIX: Check actual SEVIS column names
# ═══════════════════════════════════════════════════════════════════

print("SEVIS columns containing 'fund':")
fund_cols = [col for col in sevis_clean.columns if 'fund' in col.lower()]
for col in fund_cols:
    print(f"  • {col}")

print("\nSEVIS columns containing 'country':")
country_cols = [col for col in sevis_clean.columns if 'country' in col.lower()]
for col in country_cols:
    print(f"  • {col}")

SEVIS columns containing 'fund':
  • number_of_months_(for_funding)
  • students_personal_funds
  • funds_from_this_school
  • school_fund_type
  • funds_from_other_sources
  • school_fund_type_standardized

SEVIS columns containing 'country':
  • country_of_birth
  • country_of_citizenship
  • foreign_country
  • foreign_telephone_country_code
  • passport_country_of_issuance


In [5]:
# Check Outreach column names
print("Outreach columns after cleaning:")
for col in sorted(outreach_clean.columns):
    print(f"  • {col}")

Outreach columns after cleaning:
  • accepted_admission
  • application_agency_code
  • application_source
  • appointment_date
  • citizenship_(primary)
  • contact_number
  • deposit_paid
  • global_grad_indicator
  • i_20_in_slate
  • i_901_status
  • person_email
  • person_slu_email_address
  • program_of_interest
  • remarks
  • sevis_id
  • student_full_name
  • student_status
  • term


In [12]:
# Verify visa count using SQL syntax on the DataFrame
!pip install pandasql  # Install if not already available

import pandas as pd
from pandasql import sqldf

# Load the exported master table (or use the existing DataFrame if still in memory)
fact_student_status = pd.read_csv('fact_student_status.csv')

# Run the SQL query
result = sqldf("SELECT COUNT(*) FROM fact_student_status WHERE has_visa = 1;")
print("Number of students with visa = 1:", result.iloc[0, 0])

  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26773 sha256=493fa8ca6d936c35b558c2b29e18517b336a76c1c8ff1b741566649370d02c23
  Stored in directory: /root/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql
Number of students with visa = 1: 325
